In [1]:
import anndata as ad
import numpy as np
import pandas as pd

In [2]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

In [3]:
import perturb_lib as plib

In [4]:
import pubchempy as pcp
from tqdm import tqdm

In [5]:
import sys

import re
import time
import json
import os
import pandas as pd
from typing import Any, Optional, Dict, Callable
import pubchempy as pcp
from rdkit import Chem

import logging
from typing import List, Sequence, Any, Dict, Optional

# Init logger
FMT = '%(asctime)s | [%(levelname)s] %(message)s'
DATEFMT = '%Y-%m-%d %H:%M:%S'
formatter = logging.Formatter(fmt=FMT, datefmt=DATEFMT)

h1 = logging.StreamHandler(sys.stdout)
h1.setLevel(logging.INFO)
h1.addFilter(lambda log: log.levelno == logging.INFO)
h1.setFormatter(formatter)

h2 = logging.StreamHandler(sys.stderr)
h2.setLevel(logging.WARNING)
h2.setFormatter(formatter)

logger = logging.getLogger(__name__)
logger.propagate = False
logger.setLevel(logging.DEBUG)
logger.handlers = [h1, h2]



def is_valid_pubchem_cid(cid) -> bool:
    """Check if pubchem_cid is valid (positive integer)."""
    if cid is None:
        return False
    try:
        return int(cid) > 0
    except (ValueError, TypeError):
        return False


def is_valid_inchikey(inchikey: Optional[str]) -> bool:
    """Check if InChIKey follows standard format (XXXXXXXXXXXXXX-XXXXXXXXXX-X)."""
    if not inchikey or not isinstance(inchikey, str):
        return False
    pattern = r'^[A-Z]{14}-[A-Z]{10}-[A-Z]$'
    return bool(re.match(pattern, inchikey.strip()))


def is_valid_smiles(smiles: str) -> bool:
    """Basic SMILES validation (non-empty string with common SMILES characters)."""
    return Chem.MolFromSmiles(smiles, sanitize=True) is not None


def load_cache_from_json(cache_path: str) -> Dict[str, Optional[int]]:
    """
    Load cache dictionary from JSON file if it exists.
    
    Parameters:
    -----------
    cache_path : str
        Path to the JSON file containing the cache
        
    Returns:
    --------
    Dict[str, Optional[int]]
        Cache dictionary loaded from file, or empty dict if file doesn't exist
    """
    if os.path.exists(cache_path):
        try:
            with open(cache_path, 'r') as f:
                cache = json.load(f)
            # Convert loaded JSON values to Optional[int] type
            # JSON stores None as null, which becomes None in Python
            # Integer values are stored as numbers in JSON, which become Python int when loaded
            cache_typed = {}
            for key, value in cache.items():
                cache_typed[key] = int(value) if value is not None else None
            logger.info(f"Loaded cache from {cache_path} with {len(cache_typed)} entries")
            return cache_typed
        except Exception as e:
            logger.warning(f"Failed to load cache from {cache_path}: {e}. Starting with empty cache.")
            return {}
    else:
        logger.info(f"Cache file {cache_path} not found. Starting with empty cache.")
        return {}


def save_cache_to_json(cache: Dict[str, Optional[int]], cache_path: str) -> None:
    """
    Save cache dictionary to JSON file.
    
    Parameters:
    -----------
    cache : Dict[str, Optional[int]]
        Cache dictionary to save
    cache_path : str
        Path to save the JSON file
    """
    try:
        # Create directory if it doesn't exist
        cache_dir = os.path.dirname(cache_path)
        if cache_dir:  # Only create directory if path contains a directory
            os.makedirs(cache_dir, exist_ok=True)
        
        # Save cache directly to JSON (cache is already JSON-serializable: Dict[str, Optional[int]])
        # None values are preserved as null, integers are stored as numbers (JSON natively supports integers)
        with open(cache_path, 'w') as f:
            json.dump(cache, f, indent=2)
        logger.debug(f"Saved cache to {cache_path} with {len(cache)} entries")
    except Exception as e:
        logger.warning(f"Failed to save cache to {cache_path}: {e}")





def _fetch_pubchem_cid_with_retry(identifier: str,
                                  lookup_type: str,
                                  cache: Dict[str, Optional[int]],
                                  cache_key: str,
                                  identifier_label: str,
                                  universal_cache_key: Optional[str] = None,
                                  n_retries: int = 5) -> Optional[int]:
    """
    Helper function to fetch PubChem CID with retry logic.
    
    Parameters:
    -----------
    identifier : str
        The identifier to look up (InChIKey, SMILES, or drug name)
    lookup_type : str
        PubChem lookup type: 'inchikey', 'smiles', or 'name'
    cache : Dict[str, Optional[int]]
        Cache dictionary for storing results
    cache_key : str
        Method-specific key to use in cache dictionary
    identifier_label : str
        Label for logging (e.g., 'InChIKey', 'SMILES', 'drug name')
    universal_cache_key : Optional[str]
        Universal cache key (e.g., perturbagen name) to store result under
    n_retries : int
        Number of retries on failure
        
    Returns:
    --------
    Optional[int]
        PubChem CID or None if not found
    """
    cnt = 0
    cid = None
    while cnt < n_retries:
        try:
            compounds = pcp.get_compounds(identifier, lookup_type)
            cid = compounds[0].cid if compounds else None
            logger.debug("CID for %s '%s': %s", identifier_label, identifier, cid)
            break
        except Exception as e:
            if (isinstance(e, pcp.PubChemHTTPError) or isinstance(e, pcp.TimeoutError) or
                isinstance(e, pcp.ServerError) or isinstance(e, pcp.ServerBusyError)):
                logger.warning("PubChem lookup failed for %s '%s': %s. Retry %d.", 
                             identifier_label, identifier, str(e), cnt)
                cnt += 1
                time.sleep(5)
            else:
                logger.warning("PubChem lookup failed for %s '%s': %s", 
                             identifier_label, identifier, str(e))
                break
    
    # Store in both method-specific and universal cache keys (only if cid is not None)
    if cid is not None:
        cache[cache_key] = cid
        if universal_cache_key:
            cache[universal_cache_key] = cid
    
    return cid


def get_pubchem_cid_by_inchikey(inchikey: str, 
                                 cache: Dict[str, Optional[int]], 
                                 universal_cache_key: Optional[str] = None,
                                 n_retries: int = 5) -> Optional[int]:
    """
    Fetch PubChem CID for a given InChIKey, using cache to skip repeat lookups.
    
    Parameters:
    -----------
    inchikey : str
        InChIKey for mapping to PubChem CID
    cache : Dict[str, Optional[int]]
        A dictionary storing the mapping of identifiers to PubChem CIDs
    universal_cache_key : Optional[str]
        Universal cache key (e.g., perturbagen name) to check/store result
    n_retries : int
        The number of retries when connecting to PubChem goes wrong
        
    Returns:
    --------
    Optional[int]
        PubChem CID or None if not found
    """
    if not is_valid_inchikey(inchikey):
        return None
    
    # Check universal cache first
    if universal_cache_key and universal_cache_key in cache:
        return cache[universal_cache_key]
    
    # Check method-specific cache
    cache_key = f"inchikey:{inchikey}"
    if cache_key in cache:
        return cache[cache_key]
    
    return _fetch_pubchem_cid_with_retry(inchikey, 'inchikey', cache, cache_key, 'InChIKey', 
                                        universal_cache_key, n_retries)


def get_pubchem_cid_by_chembl(chembl_id: str,
                              cache: Dict[str, Optional[int]],
                              universal_cache_key: Optional[str] = None,
                              n_retries: int = 5) -> Optional[int]:
    """
    Fetch PubChem CID for a given ChEMBL ID using PubChem name search.

    Parameters:
    -----------
    chembl_id : str
        ChEMBL identifier (e.g. 'CHEMBL1380')
    cache : Dict[str, Optional[int]]
        A dictionary storing the mapping of identifiers to PubChem CIDs
    universal_cache_key : Optional[str]
        Universal cache key (e.g., perturbagen name) to check/store result
    n_retries : int
        The number of retries when connecting to PubChem goes wrong

    Returns:
    --------
    Optional[int]
        PubChem CID or None if not found
    """
    if pd.isna(chembl_id) or not chembl_id or not isinstance(chembl_id, str):
        return None

    chembl_id = chembl_id.strip()

    # Check universal cache first
    if universal_cache_key and universal_cache_key in cache:
        return cache[universal_cache_key]

    # Check method-specific cache
    cache_key = f"chembl:{chembl_id}"
    if cache_key in cache:
        return cache[cache_key]

    cnt = 0
    cid = None
    while cnt < n_retries:
        try:
            cids = pcp.get_cids(chembl_id, namespace="name")
            cid = cids[0] if cids else None
            logger.debug("CID for ChEMBL ID '%s': %s", chembl_id, cid)
            break
        except Exception as e:
            if (isinstance(e, pcp.PubChemHTTPError) or isinstance(e, pcp.TimeoutError) or
                isinstance(e, pcp.ServerError) or isinstance(e, pcp.ServerBusyError)):
                logger.warning("PubChem lookup failed for ChEMBL ID '%s': %s. Retry %d.",
                             chembl_id, str(e), cnt)
                cnt += 1
                time.sleep(5)
            else:
                logger.warning("PubChem lookup failed for ChEMBL ID '%s': %s", chembl_id, str(e))
                break

    if cid is not None:
        cache[cache_key] = cid
        if universal_cache_key:
            cache[universal_cache_key] = cid

    return cid


def get_pubchem_cid_by_smiles(smiles: str, 
                              cache: Dict[str, Optional[int]], 
                              universal_cache_key: Optional[str] = None,
                              n_retries: int = 5) -> Optional[int]:
    """
    Fetch PubChem CID for a given SMILES string, using cache to skip repeat lookups.
    
    Parameters:
    -----------
    smiles : str
        SMILES string for mapping to PubChem CID
    cache : Dict[str, Optional[int]]
        A dictionary storing the mapping of identifiers to PubChem CIDs
    universal_cache_key : Optional[str]
        Universal cache key (e.g., perturbagen name) to check/store result
    n_retries : int
        The number of retries when connecting to PubChem goes wrong
        
    Returns:
    --------
    Optional[int]
        PubChem CID or None if not found
    """
    if not is_valid_smiles(smiles):
        return None
    
    # Check universal cache first
    if universal_cache_key and universal_cache_key in cache:
        return cache[universal_cache_key]
    
    # Check method-specific cache
    cache_key = f"smiles:{smiles}"
    if cache_key in cache:
        return cache[cache_key]
    
    return _fetch_pubchem_cid_with_retry(smiles, 'smiles', cache, cache_key, 'SMILES', 
                                        universal_cache_key, n_retries)


def get_pubchem_cid_by_name(drug_name: str, 
                            cache: Dict[str, Optional[int]], 
                            universal_cache_key: Optional[str] = None,
                            n_retries: int = 5) -> Optional[int]:
    """
    Fetch PubChem CID for a given drug name, using cache to skip repeat lookups.
    
    Parameters:
    -----------
    drug_name : str
        Drug name for mapping to PubChem CID
    cache : Dict[str, Optional[int]]
        A dictionary storing the mapping of identifiers to PubChem CIDs
    universal_cache_key : Optional[str]
        Universal cache key (e.g., perturbagen name) to check/store result.
        If None, uses drug_name as universal key.
    n_retries : int
        The number of retries when connecting to PubChem goes wrong
        
    Returns:
    --------
    Optional[int]
        PubChem CID or None if not found
    """
    if pd.isna(drug_name) or not drug_name:
        return None
    
    # Use drug_name as universal key if not provided
    if universal_cache_key is None:
        universal_cache_key = drug_name
    
    # Check universal cache first
    if universal_cache_key in cache:
        return cache[universal_cache_key]
    
    # Check method-specific cache
    cache_key = f"name:{drug_name}"
    if cache_key in cache:
        return cache[cache_key]
    
    return _fetch_pubchem_cid_with_retry(drug_name, 'name', cache, cache_key, 'drug name', 
                                        universal_cache_key, n_retries)


def lookup_pubchem_cids(df: pd.DataFrame,
                           cache: Dict[str, Optional[int]],
                           pert_id_col: Optional[str] = 'pert_id',
                           drug_col: str = 'perturbagen',
                           pubchem_cid_col: str = 'pubchem_cid',
                           inchikey_col: str = 'inchi_key',
                           chembl_col: str = 'molecule_chembl_id',
                           smiles_col: str = 'canonical_smiles',
                           cache_path: Optional[str] = None,
                           manual_mapping_func: Optional[Callable[[], Dict]] = None,
                           manual_mapping_by_drug_name: bool = True,
                           dataset_key: Optional[str] = None,
                           request_delay_s: float = 0.2) -> pd.DataFrame:
    """
    Add or update 'pubchem_cid' column in a dataframe.
    
    Uses multiple strategies in order of preference:
    1. Use universal cache key (pert_id or perturbagen) if present
    2. Use existing valid pubchem_cid if present
    3. Lookup by InChIKey if available and valid
    4. Lookup by ChEMBL ID (via PubChem name search) if available
    5. Lookup by SMILES if available and valid
    6. Use manual mapping from manual_mapping_func by pert id or drug name
    7. Lookup by drug name (perturbagen) 
    
    Uses universal cache keys (pert_id or perturbagen) to avoid redundant lookups
    when the same compound is identified by different methods.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to add pubchem_cid column to
    cache : Dict[str, Optional[int]]
        A dictionary storing the mapping of identifiers to PubChem CIDs.
        Will be updated with new entries during processing.
    pert_id_col : Optional[str], default='pert_id'
        Column name for perturbation ID to use as universal cache key.
        If None, uses drug_col (perturbagen) as universal cache key.
    drug_col : str, default='perturbagen'
        Column name containing drug names (used as universal cache key if pert_id_col is None)
    pubchem_cid_col : str, default='pubchem_cid'
        Column name for PubChem CID (will be created/updated)
    inchikey_col : str, default='inchi_key'
        Column name for InChIKey
    chembl_col : str, default='molecule_chembl_id'
        Column name for ChEMBL ID (used via PubChem name search)
    smiles_col : str, default='canonical_smiles'
        Column name for SMILES
    cache_path : Optional[str], default=None
        Path to JSON file for persistent cache storage.
        If provided, cache will be loaded from this file at start and saved periodically.
    manual_mapping_func : Optional[Callable[[], Dict]], default=None
        Function that returns manual PubChem CID mappings. If provided, should return
        either a dict directly (e.g., {'drug_name': cid}) or a dict with dataset keys
        (e.g., {'dataset_name': {'drug_name': cid}}). If None, no manual mapping is used.
    dataset_key : Optional[str], default=None
        Key to extract from the dict returned by manual_mapping_func if it returns
        a nested dict structure. If None and manual_mapping_func returns a nested dict,
        uses the first dataset in that mapping.
    request_delay_s : float, default=0.2
        Delay in seconds at the end of each compound iteration (fair-use throttling).
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with updated pubchem_cid column
    """
    
    if df is None:
        raise Exception("The pseudobulk dataset is empty")
    
    df = df.copy()
    # Load cache from file if path is provided
    if cache_path:
        file_cache = load_cache_from_json(cache_path)
        merged_cache = {**file_cache, **cache}.copy()
        cache.clear()
        cache.update(merged_cache)
    
    # Initialize pubchem_cid column if it doesn't exist
    if pubchem_cid_col not in df.columns:
        df[pubchem_cid_col] = None
    
    # Get manual mappings
    if manual_mapping_func is not None:
        sm2pubchem = manual_mapping_func()
        # Handle both flat dicts and nested dicts
        if isinstance(sm2pubchem, dict):
            if dataset_key is not None:
                # Extract specific key from nested dict
                manual_mapping = sm2pubchem.get(dataset_key, {})
            elif len(sm2pubchem) == 1:
                # If single key, use it automatically
                only_value = list(sm2pubchem.values())[0]
                manual_mapping = only_value if isinstance(only_value, dict) else sm2pubchem
            else:
                # Check if it's a nested dict (values are dicts) or flat dict (values are ints)
                first_value = list(sm2pubchem.values())[0] if sm2pubchem else None
                if isinstance(first_value, dict):
                    # Nested dict but no dataset_key specified - use first key as fallback
                    manual_mapping = first_value
                else:
                    # Flat dict with drug_name: cid mappings
                    manual_mapping = sm2pubchem
        else:
            manual_mapping = {}
    else:
        manual_mapping = {}
    
    # Process each row to determine best CID source
    cids = []
    iteration_count = 0
    n_compounds = len(df)
    logger.info(f"Processing {n_compounds} compounds")
    for idx, row in df.iterrows():
        iteration_count += 1
        cid = None
        stored_cid = False
        
        # Determine universal cache key (pert_id or perturbagen)
        if pert_id_col and pert_id_col in row and pd.notna(row[pert_id_col]):
            universal_key = str(row[pert_id_col]).strip()
        elif drug_col in row and pd.notna(row[drug_col]):
            universal_key = str(row[drug_col]).strip()
        else:
            universal_key = None
        
        # Check universal cache first
        if universal_key and universal_key in cache:
            cid = cache[universal_key]
            stored_cid = True
        
        # Strategy 1: Use existing valid pubchem_cid (if CID not found yet)
        if cid is None and pubchem_cid_col in row and is_valid_pubchem_cid(row[pubchem_cid_col]):
            cid = int(row[pubchem_cid_col])
            # Store in universal cache
            if universal_key:
                cache[universal_key] = cid
            stored_cid = True
        
        # Strategy 2: Lookup by InChIKey (if CID not found yet)
        if cid is None and inchikey_col in row and pd.notna(row[inchikey_col]):
            inchikey = str(row[inchikey_col]).strip()
            if is_valid_inchikey(inchikey):
                cid = get_pubchem_cid_by_inchikey(inchikey, cache, universal_key)

        # Strategy 3: Lookup by ChEMBL ID (if CID not found yet)
        if cid is None and chembl_col in row and pd.notna(row[chembl_col]):
            chembl_id = str(row[chembl_col]).strip()
            if chembl_id:
                cid = get_pubchem_cid_by_chembl(chembl_id, cache, universal_key)

        # Strategy 4: Lookup by SMILES (if CID not found yet)
        if cid is None and smiles_col in row and pd.notna(row[smiles_col]):
            smiles = str(row[smiles_col]).strip()
            if is_valid_smiles(smiles):
                cid = get_pubchem_cid_by_smiles(smiles, cache, universal_key)

        # Strategy 5: Manual mapping:
        if not manual_mapping_by_drug_name:
            if cid is None and pert_id_col in row and pd.notna(row[pert_id_col]):
                pert_id = str(row[pert_id_col]).strip()
                if pert_id in manual_mapping:
                    cid = manual_mapping[pert_id]
                    # Store in universal cache
                    if universal_key:
                        cache[universal_key] = cid
                    stored_cid = True

        else:
            if cid is None and drug_col in row and pd.notna(row[drug_col]):
                drug_name = str(row[drug_col]).strip()
                if drug_name in manual_mapping:
                    cid = manual_mapping[drug_name]
                    # Store in universal cache
                    if universal_key:
                        cache[universal_key] = cid
                    stored_cid = True
        
        # Strategy 6: Lookup by drug name (if CID not found yet)
        if cid is None and drug_col in row and pd.notna(row[drug_col]):
            drug_name = str(row[drug_col]).strip()
            cid = get_pubchem_cid_by_name(drug_name, cache, universal_key)
        
                
        
        cids.append(cid)
        
        # Log progress every 50 compounds
        if iteration_count % 50 == 0:
            n_mapped_so_far = sum(1 for c in cids if c is not None)
            logger.info(f"Processed {iteration_count}/{n_compounds} compounds ({n_mapped_so_far} mapped so far)")
        
        # Save cache every 500 iterations if cache_path is provided
        if cache_path and iteration_count % 500 == 0:
            save_cache_to_json(cache, cache_path)
        
        if (request_delay_s > 0) and (not stored_cid):
            time.sleep(request_delay_s)
    
    # Final save of cache if cache_path is provided
    if cache_path:
        save_cache_to_json(cache, cache_path)
    
    # Update pubchem_cid column
    df_updated = df.copy()
    df_updated[pubchem_cid_col] = cids
    
    n_mapped = df_updated[pubchem_cid_col].notna().sum()
    logger.info(f"Mapped {n_mapped} out of {len(df)} compounds to PubChem CIDs")
    
    return df_updated

In [6]:
model = plib.load_trained_model('../../../perturblib/.plib_cache/results/lincs_paper_lpm/LPM_9bad9756f740b28a/seed_13/model.pt')

In [7]:
df_pert = model.vocab.perturb_vocab.to_pandas()

In [8]:
embeddings = model.perturb_embedding_layer.weight.numpy().astype(np.float64)

In [9]:
df_dili = pd.read_csv('../../../DILI_from_web.csv')

In [10]:
pd.options.display.max_columns = 100

In [11]:
df_dili = df_dili.rename(columns={'Unnamed: 0': 'pert_id'})

In [12]:
df_dili['pubchem_cid'] = None

In [13]:
'''
dili_compounds = lookup_pubchem_cids(
        df_dili, 
        cache = {}, 
        pert_id_col = 'pert_id',
        drug_col = 'compound_name',
        pubchem_cid_col = 'pubchem_cid',
        inchikey_col = None,
        chembl_col = 'molecule_chembl_id',
        smiles_col = 'smiles',
        cache_path='../../cache_dili.json',
        manual_mapping_func = None,
        manual_mapping_by_drug_name = False,
        dataset_key = 'dili',
    )

dili_compounds.to_csv('../../dili_pubchem.csv', index=False)
'''


"\ndili_compounds = lookup_pubchem_cids(\n        df_dili, \n        cache = {}, \n        pert_id_col = 'pert_id',\n        drug_col = 'compound_name',\n        pubchem_cid_col = 'pubchem_cid',\n        inchikey_col = None,\n        chembl_col = 'molecule_chembl_id',\n        smiles_col = 'smiles',\n        cache_path='../../cache_dili.json',\n        manual_mapping_func = None,\n        manual_mapping_by_drug_name = False,\n        dataset_key = 'dili',\n    )\n\ndili_compounds.to_csv('../../dili_pubchem.csv', index=False)\n"

In [14]:
dili_compounds = pd.read_csv('../../dili_pubchem.csv')

In [15]:
compoundinfo_df = pd.read_csv('../../compoundinfo_beta.txt', delimiter="\t", low_memory=False)
compoundinfo_df = compoundinfo_df[~compoundinfo_df.duplicated(['cmap_name', 'canonical_smiles'])]

In [16]:
df_pert['symbol_'] = df_pert['symbol'].str.replace('-10uM', '')
df_sm = df_pert[~(df_pert['symbol'].str.contains('CRISPR'))]
compoundinfo_df = compoundinfo_df[compoundinfo_df['cmap_name'].isin(df_sm['symbol_'])]

In [17]:
df_sm[~df_sm['symbol_'].isin(compoundinfo_df['cmap_name'])]

,symbol,code,symbol_
0,Control,0,Control
1,BRD3308-10uM,1,BRD3308
2762,BRD2492,2762,BRD2492


In [18]:
'''
df_compounds = lookup_pubchem_cids(
        compoundinfo_df, 
        cache={}, 
        pert_id_col='pert_id',
        drug_col='cmap_name',
        cache_path='../../cache.json',
        manual_mapping_func=None,
        manual_mapping_by_drug_name=False,
        dataset_key='l1000'
    )
df_compounds.to_csv('../../df_compounds.csv')
'''
df_compounds = pd.read_csv('../../df_compounds.csv')

[23:04:23] SMILES Parse Error: syntax error while parsing: restricted
[23:04:23] SMILES Parse Error: check for mistakes around position 1:
[23:04:23] restricted
[23:04:23] ^
[23:04:26] SMILES Parse Error: Failed parsing SMILES 'restricted' for input: 'restricted'


2026-03-12 23:04:29 | [INFO] Processed 50/6254 compounds (49 mapped so far)
2026-03-12 23:05:04 | [INFO] Processed 100/6254 compounds (99 mapped so far)
2026-03-12 23:05:38 | [INFO] Processed 150/6254 compounds (149 mapped so far)
2026-03-12 23:06:12 | [INFO] Processed 200/6254 compounds (199 mapped so far)
2026-03-12 23:06:45 | [INFO] Processed 250/6254 compounds (249 mapped so far)
2026-03-12 23:07:19 | [INFO] Processed 300/6254 compounds (299 mapped so far)
2026-03-12 23:07:52 | [INFO] Processed 350/6254 compounds (349 mapped so far)
2026-03-12 23:08:25 | [INFO] Processed 400/6254 compounds (399 mapped so far)
2026-03-12 23:08:58 | [INFO] Processed 450/6254 compounds (449 mapped so far)


[23:08:59] SMILES Parse Error: syntax error while parsing: restricted
[23:08:59] SMILES Parse Error: check for mistakes around position 1:
[23:08:59] restricted
[23:08:59] ^
[23:09:03] SMILES Parse Error: Failed parsing SMILES 'restricted' for input: 'restricted'
2026-03-12 23:09:25 | [WARNING] PubChem lookup failed for SMILES 'CCP(CC)(CC)=[Au]SC1CC(OC(C)=O)C(OC(C)=O)C(OC(C)=O)C1OC(C)=O': PubChem HTTP Error 400 PUGREST.BadRequest: Unable to standardize the given structure - perhaps some special characters need to be escaped or data packed in a MIME form? (error: , status: 400, output: Caught ncbi::CException: Standardization failed, Output Log:, Record 1: Warning: Detected illegal valence for element "Au": 1 sigma bonds, 1 pi bonds, 0 charge, Record 1: Warning: Compound failed verification during "standardize deposited compound", , PubChem Warning; class: Invalid Chemical Structure; label: Structure Standardization Issues; message: Detected illegal valence for element "Au": 1 sigma bon

2026-03-12 23:10:04 | [INFO] Processed 500/6254 compounds (499 mapped so far)


2026-03-12 23:10:04 | [WARNING] Failed to save cache to ./: [Errno 21] Is a directory: './'


...

2026-03-13 00:14:05 | [INFO] Processed 5950/6254 compounds (5530 mapped so far)
2026-03-13 00:14:39 | [INFO] Processed 6000/6254 compounds (5580 mapped so far)


2026-03-13 00:14:39 | [WARNING] Failed to save cache to ./: [Errno 21] Is a directory: './'


2026-03-13 00:15:15 | [INFO] Processed 6050/6254 compounds (5630 mapped so far)
2026-03-13 00:15:56 | [INFO] Processed 6100/6254 compounds (5680 mapped so far)
2026-03-13 00:16:29 | [INFO] Processed 6150/6254 compounds (5730 mapped so far)
2026-03-13 00:17:05 | [INFO] Processed 6200/6254 compounds (5780 mapped so far)
2026-03-13 00:17:40 | [INFO] Processed 6250/6254 compounds (5830 mapped so far)


2026-03-13 00:17:43 | [WARNING] Failed to save cache to ./: [Errno 21] Is a directory: './'


2026-03-13 00:17:44 | [INFO] Mapped 5834 out of 6254 compounds to PubChem CIDs


In [19]:
df_compounds = df_compounds.drop(columns=['Unnamed: 0'])

In [21]:
dili_compounds['retrieved_smiles'] = [pcp.Compound.from_cid(int(x)).connectivity_smiles if not pd.isna(x) else x for x in tqdm(dili_compounds['pubchem_cid'])]

100%|██████████| 2472/2472 [15:13<00:00,  2.71it/s]


In [22]:
df_compounds.drop_duplicates(['cmap_name', 'pubchem_cid']).assign(key=df_compounds.drop_duplicates(['cmap_name', 'pubchem_cid'])["pubchem_cid"].fillna("__NA2__"))[['cmap_name', 'pubchem_cid', 'key']]

,cmap_name,pubchem_cid,key
0,BRD-A18795974,1219.0,1219.0
1,BRD-A77577770,462.0,462.0
2,SPP-301,9912992.0,9912992.0
3,PRT-062070,44595079.0,44595079.0
4,L-ergothioneine,165404.0,165404.0
...,...,...,...
6249,deguelin,107935.0,107935.0
6250,BRD-A61599461,50897816.0,50897816.0
6251,KN-93,5312122.0,5312122.0
6252,TAS-301,9885087.0,9885087.0


In [23]:
dili_compounds_merged = (
    dili_compounds.assign(key=dili_compounds["pubchem_cid"].fillna("__NA1__"))
       .merge(df_compounds.drop_duplicates(['cmap_name', 'pubchem_cid']).assign(key=df_compounds.drop_duplicates(['cmap_name', 'pubchem_cid'])["pubchem_cid"].fillna("__NA2__"))[['cmap_name', 'pubchem_cid', 'key']], on="key", how="left")
       .replace({"__NA1__": pd.NA, "__NA2__": pd.NA})
)

In [24]:
dili_compounds_merged[dili_compounds_merged['pert_id'].duplicated(keep=False)]

,pert_id,LTKBID,compound_name,DILIrank,label_section,severity_class,DILIst,roa,source,livertox_score,livertox_primary_classification,livertox_secondary_classification,molecule_chembl_id,atc_classifications,availability_type,black_box_warning,chemical_probe,chirality,dosed_ingredient,first_approval,first_in_class,helm_notation,inorganic_flag,max_phase,molecule_type,natural_product,oral,orphan,parenteral,polymer_flag,pref_name,prodrug,structure_type,therapeutic_flag,topical,usan_stem,usan_stem_definition,usan_substem,usan_year,veterinary,withdrawn_flag,smiles,mw,alogp,warning_class,warning_country,warning_description,warning_type,withdrawn_reason,target,pathway,information,CAS_number,DMSO_mM_solubility,DILI_label,DILI_label_binary,DILI_label_section,livertox_mechanism,livertox_information,livertox_updated,livertox_iDILI,livertox_mechanism_summary,pubchem_cid_x,retrieved_smiles,key,cmap_name,pubchem_cid_y
881,Scopolamine,LT01974,Scopolamine,No-DILI-Concern,No match,0.0,NaN,Intravenous,DILIrank,E,Gastrointestinal,Antiemetic,CHEMBL569713,S01FA02; A04AD01; A04AD51; N05CM05,1.0,0.0,0.0,1.0,True,1979.0,0.0,NaN,0.0,4.0,Small molecule,1.0,False,0.0,False,0.0,SCOPOLAMINE,0.0,MOL,True,True,NaN,NaN,NaN,NaN,0.0,False,CN1[C@@H]2C[C@@H](OC(=O)[C@H](CO)c3ccccc3)C[C@...,303.36,0.92,NaN,NaN,NaN,NaN,NaN,AChR,Neuronal Signaling,"Scopolamine HBr (LSM-4015,NSC 61806,(-)-Scopol...",114-49-8,197.78,No DILI,No DILI,No DILI,NaN,NaN,NaN,NaN,NaN,3000322.0,CN1C2CC(CC1C3C2O3)OC(=O)C(CO)C4=CC=CC=C4,3000322.0,BRD-A93048969,3000322.0
882,Scopolamine,LT01974,Scopolamine,No-DILI-Concern,No match,0.0,NaN,Intravenous,DILIrank,E,Gastrointestinal,Antiemetic,CHEMBL569713,S01FA02; A04AD01; A04AD51; N05CM05,1.0,0.0,0.0,1.0,True,1979.0,0.0,NaN,0.0,4.0,Small molecule,1.0,False,0.0,False,0.0,SCOPOLAMINE,0.0,MOL,True,True,NaN,NaN,NaN,NaN,0.0,False,CN1[C@@H]2C[C@@H](OC(=O)[C@H](CO)c3ccccc3)C[C@...,303.36,0.92,NaN,NaN,NaN,NaN,NaN,AChR,Neuronal Signaling,"Scopolamine HBr (LSM-4015,NSC 61806,(-)-Scopol...",114-49-8,197.78,No DILI,No DILI,No DILI,NaN,NaN,NaN,NaN,NaN,3000322.0,CN1C2CC(CC1C3C2O3)OC(=O)C(CO)C4=CC=CC=C4,3000322.0,scopolamine,3000322.0


In [25]:
dili_compounds_merged = dili_compounds_merged.drop(index=881).reset_index(drop=True)

In [26]:
dili_compounds_merged[dili_compounds_merged['pert_id'].duplicated(keep=False)]

,pert_id,LTKBID,compound_name,DILIrank,label_section,severity_class,DILIst,roa,source,livertox_score,livertox_primary_classification,livertox_secondary_classification,molecule_chembl_id,atc_classifications,availability_type,black_box_warning,chemical_probe,chirality,dosed_ingredient,first_approval,first_in_class,helm_notation,inorganic_flag,max_phase,molecule_type,natural_product,oral,orphan,parenteral,polymer_flag,pref_name,prodrug,structure_type,therapeutic_flag,topical,usan_stem,usan_stem_definition,usan_substem,usan_year,veterinary,withdrawn_flag,smiles,mw,alogp,warning_class,warning_country,warning_description,warning_type,withdrawn_reason,target,pathway,information,CAS_number,DMSO_mM_solubility,DILI_label,DILI_label_binary,DILI_label_section,livertox_mechanism,livertox_information,livertox_updated,livertox_iDILI,livertox_mechanism_summary,pubchem_cid_x,retrieved_smiles,key,cmap_name,pubchem_cid_y


In [27]:
dili_compounds_merged = dili_compounds_merged.rename(columns={'pubchem_cid_x': 'pubchem_cid'}).drop(columns=['pubchem_cid_y', 'key'])

In [28]:
df_merged = dili_compounds_merged.merge(df_sm, left_on='cmap_name', right_on='symbol_', how='left')

In [29]:
def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smiles in smiles_list:
        try:
            mol0 = Chem.MolFromSmiles(smiles[0])
        except:
            mol0 = None
        try:
            mol1 = Chem.MolFromSmiles(smiles[1])
        except:
            mol1 = None
        if mol0 is not None:
            fp = gen.GetFingerprint(mol0)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
        elif mol1 is not None:
            fp = gen.GetFingerprint(mol1)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
        else:
            fps.append(None)
    return fps

df_merged['ECFP:2'] = smiles_to_fingerprints(df_merged[['smiles', 'retrieved_smiles']].values)

In [30]:
emb_list = []
for c in df_merged['code']:
    if pd.isna(c):
        emb_list.append(None)
    else:
        emb_list.append(embeddings[int(c)])

In [31]:
df_merged['LPM_emb'] = emb_list

In [32]:
df_merged[df_merged['pert_id'] != df_dili['pert_id']]

,pert_id,LTKBID,compound_name,DILIrank,label_section,severity_class,DILIst,roa,source,livertox_score,livertox_primary_classification,livertox_secondary_classification,molecule_chembl_id,atc_classifications,availability_type,black_box_warning,chemical_probe,chirality,dosed_ingredient,first_approval,first_in_class,helm_notation,inorganic_flag,max_phase,molecule_type,natural_product,oral,orphan,parenteral,polymer_flag,pref_name,prodrug,structure_type,therapeutic_flag,topical,usan_stem,usan_stem_definition,usan_substem,usan_year,veterinary,withdrawn_flag,smiles,mw,alogp,warning_class,warning_country,warning_description,warning_type,withdrawn_reason,target,pathway,information,CAS_number,DMSO_mM_solubility,DILI_label,DILI_label_binary,DILI_label_section,livertox_mechanism,livertox_information,livertox_updated,livertox_iDILI,livertox_mechanism_summary,pubchem_cid,retrieved_smiles,cmap_name,symbol,code,symbol_,ECFP:2,LPM_emb


In [39]:
#df_merged.to_pickle("../../dili.pkl")